# 03 Encoding Representations

This notebook runs the 5 encoders specified in the project plan:
1. TF-IDF + SVD (300D)
2. Word2Vec (300D)
3. GloVe (300D)
4. SBERT (384D)
5. BGE-large (1024D)

Embeddings are saved to `data/embeddings/`.

## Encoder Comparison: Pros and Cons

Before generating the embeddings, here is a theoretical comparison of the models we are evaluating:

| Encoder | Category | Pros | Cons |
|:---|:---|:---|:---|
| **TF-IDF + SVD** | Sparse / Linear | Fast, interpretable, great baseline. | Ignores word order and semantics (synonyms). |
| **Word2Vec** | Static Dense | Captures local semantic relationships (linear). | Out-of-vocabulary issues; cannot handle polysemy (context). |
| **GloVe** | Static Dense | Captures global corpus statistics well. | Similar to Word2Vec, it generates static, context-free vectors. |
| **SBERT** | Contextual Dense | Excellent for semantic similarity and clustering. | Slower to compute; requires GPU for large datasets. |
| **BGE-large** | Contextual Dense | State-of-the-Art for retrieval; handles complex context. | Very high dimensional (1024D); computationally expensive. |


In [ ]:
import pandas as pd
import numpy as np
import os
import sys
sys.path.append('..')

from src.encoders import TFIDFEncoder, W2VEncoder, GloVeEncoder, SBERTEncoder, BGEEncoder
from src.preprocess import preprocess_data

## 1. Load and Preprocess Data

In [ ]:
# Load processed data if available, otherwise load raw and preprocess
processed_path = '../data/processed/cleaned_reviews.parquet'
raw_dir = '../data/raw/'

if os.path.exists(processed_path):
    print("Loading processed data...")
    df = pd.read_parquet(processed_path)
else:
    from glob import glob
    latest_raw = max(glob(os.path.join(raw_dir, "*.jsonl")), key=os.path.getctime)
    print(f"Loading raw data from {latest_raw}...")
    df = pd.read_json(latest_raw, lines=True)
    df = preprocess_data(df)
    
print(f"Data shape: {df.shape}")

## 2. Initialize and Run Encoders

We will save each embedding matrix as a `.npy` file.

In [ ]:
embedding_dir = '../data/embeddings/'
os.makedirs(embedding_dir, exist_ok=True)
texts = df['cleaned_text'].tolist()

### 2.1 TF-IDF + SVD

In [ ]:
tfidf_encoder = TFIDFEncoder(n_components=300)
tfidf_embeddings = tfidf_encoder.encode(texts)
np.save(os.path.join(embedding_dir, 'tfidf.npy'), tfidf_embeddings)
print("Saved TF-IDF embeddings.")

### 2.2 Word2Vec (Trained on Corpus)

In [ ]:
tokenized_texts = [text.split() for text in texts]
w2v_encoder = W2VEncoder(vector_size=300)
w2v_embeddings = w2v_encoder.encode(tokenized_texts)
np.save(os.path.join(embedding_dir, 'w2v.npy'), w2v_embeddings)
print("Saved Word2Vec embeddings.")

### 2.3 GloVe (840B 300D)

> **Note:** Requires `glove.840B.300d.txt` in the root or specified path. If missing, this will skip or error.

In [ ]:
glove_path = '../glove.840B.300d.txt'
if os.path.exists(glove_path):
    glove_encoder = GloVeEncoder(glove_path=glove_path)
    glove_embeddings = glove_encoder.encode(tokenized_texts)
    np.save(os.path.join(embedding_dir, 'glove.npy'), glove_embeddings)
    print("Saved GloVe embeddings.")
else:
    print("GloVe file not found. Skipping.")

### 2.4 SBERT (MiniLM)

In [ ]:
sbert_encoder = SBERTEncoder()
sbert_embeddings = sbert_encoder.encode(texts)
np.save(os.path.join(embedding_dir, 'sbert.npy'), sbert_embeddings)
print("Saved SBERT embeddings.")

### 2.5 BGE-large

In [ ]:
bge_encoder = BGEEncoder()
bge_embeddings = bge_encoder.encode(texts)
np.save(os.path.join(embedding_dir, 'bge.npy'), bge_embeddings)
print("Saved BGE embeddings.")

## 3. High-Level Embedding Concepts (Demonstration)

Let's visualize some of the key theoretical differences between static and contextual embeddings.

### 3.1 Static Embeddings: Word Algebra
In static embeddings like GloVe or Word2Vec, semantic relationships are encoded as linear transformations in the vector space. The famous example is `King - Man + Woman = Queen`.

*Note: This requires the GloVe embeddings to be loaded. We use a simple cosine similarity search to find the closest word to the resulting vector.*

In [ ]:
import numpy as np
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

if 'glove_encoder' in locals() and hasattr(glove_encoder, 'embeddings'):
    glove_dict = glove_encoder.embeddings
    if all(w in glove_dict for w in ['king', 'man', 'woman']):
        # King - Man + Woman
        vec = glove_dict['king'] - glove_dict['man'] + glove_dict['woman']
        
        # Find closest words (excluding the inputs)
        similarities = {}
        for word, w_vec in glove_dict.items():
            if word not in ['king', 'man', 'woman']:
                similarities[word] = cosine_sim(vec, w_vec)
        
        # Sort by highest similarity
        top_words = sorted(similarities.items(), key=lambda item: item[1], reverse=True)[:5]
        print("Closest words to [King - Man + Woman]:")
        for w, sim in top_words:
            print(f"{w}: {sim:.4f}")
else:
    print("GloVe encoder not loaded or words missing in vocabulary.")

### 3.2 The Polysemy Problem (Static vs Contextual)
Static embeddings fail at **polysemy** (words with multiple meanings, like "bank"). They only have one vector for "bank". Contextual models like SBERT generate embeddings for the whole sentence, implicitly understanding the context of the word.

In [ ]:
if 'sbert_encoder' in locals():
    sentences = [
        "I sat by the river bank and watched the water.",  # River context
        "The ducks swam near the muddy bank.",             # River context
        "I deposited my paycheck at the bank.",            # Financial context
        "The bank vault was highly secured."               # Financial context
    ]
    
    # Encode with Contextual SBERT
    sbert_vecs = sbert_encoder.encode(sentences, batch_size=4)
    
    print("\nSBERT (Contextual) Similarities:")
    print(f"River 1 vs River 2:     {cosine_sim(sbert_vecs[0], sbert_vecs[1]):.4f}")
    print(f"Finance 1 vs Finance 2: {cosine_sim(sbert_vecs[2], sbert_vecs[3]):.4f}")
    print(f"River 1 vs Finance 1:   {cosine_sim(sbert_vecs[0], sbert_vecs[2]):.4f}  <-- Notice how low this is!")
    
    print("\nStatic Embedding Failure:")
    print("In Word2Vec or GloVe, the word 'bank' in sentence 1 and sentence 3 has the EXACT SAME VECTOR.")
else:
    print("SBERT encoder not loaded.")